# task_aggregate

> This task aggregates the downloaded data into spatial and temporal averages. It uses xarray to compute summary statistics over the specified time period and spatial region. The aggregation is done diurnally, so we will fetch the sundown and sunrise times for the region and use them to compute the diurnal averages.

In [31]:
#| default_exp task_aggregate

In [32]:
#| hide
from nbdev.showdoc import *

In [33]:
#| export

from era5_sandbox.config import BLD, data_catalog
from era5_sandbox.core import ClimateDataFileHandler
from pyprojroot import here
import xarray as xr

To do diurnal aggregation, we will need to fetch the sundown and sunrise times for the region and use them to compute the diurnal averages. We will use the [astral library](https://astral.readthedocs.io/en/latest/) to get the sunrise and sunset times for the specified latitude and longitude. The aggregation will be done using xarray, which allows us to compute the mean over the specified time period and spatial region.



In [47]:
from pyparsing import C


eg_file = BLD / "2024_11_nepal.nc"
with ClimateDataFileHandler(eg_file) as handler:
    ds_path = handler.get_dataset("instant")

    ds = xr.open_dataset(ds_path, chunks = {"valid_time": 24})

/tmp/ipykernel_378201/281129892.py:8: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 24. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(ds_path, chunks = {"valid_time": 24})


In [48]:
ds

<xarray.Dataset> Size: 51MB
Dimensions:     (valid_time: 720, latitude: 49, longitude: 91)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 6kB 2024-11-01 ... 2024-11-30T23:...
  * latitude    (latitude) float64 392B 30.8 30.7 30.6 30.5 ... 26.2 26.1 26.0
  * longitude   (longitude) float64 728B 79.6 79.7 79.8 79.9 ... 88.4 88.5 88.6
    expver      (valid_time) <U4 12kB dask.array<chunksize=(24,), meta=np.ndarray>
Data variables:
    d2m         (valid_time, latitude, longitude) float32 13MB dask.array<chunksize=(24, 49, 91), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 13MB dask.array<chunksize=(24, 49, 91), meta=np.ndarray>
    tp          (valid_time, latitude, longitude) float32 13MB dask.array<chunksize=(24, 49, 91), meta=np.ndarray>
    swvl1       (valid_time, latitude, longitude) float32 13MB dask.array<chunksize=(24, 49, 91), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-07-30T13:24 GRIB to CDM+CF via cfgrib-0.9.1...

We can see the astral library in action below:

In [50]:
#| export
from astral import Observer, sun
import pandas as pd
import numpy as np

In [51]:
# get the location of a datapoint in the dataset
lat, long = ds.coords["latitude"].values[0], ds.coords["longitude"].values[0]
time = ds['valid_time'].values[0]
dt = pd.to_datetime(time, utc=True)

In [52]:
dt

Timestamp('2024-11-01 00:00:00+0000', tz='UTC')

In [53]:
observer = Observer(latitude=lat, longitude=long, elevation=0)
sun_info = sun.sun(observer, date=dt)
sun_info

{'dawn': datetime.datetime(2024, 11, 1, 0, 31, 27, 1928, tzinfo=datetime.timezone.utc),
 'sunrise': datetime.datetime(2024, 11, 1, 0, 56, 50, 466902, tzinfo=datetime.timezone.utc),
 'noon': datetime.datetime(2024, 11, 1, 6, 25, 7, tzinfo=datetime.timezone.utc),
 'sunset': datetime.datetime(2024, 11, 1, 11, 53, 2, 485384, tzinfo=datetime.timezone.utc),
 'dusk': datetime.datetime(2024, 11, 1, 12, 18, 25, 539578, tzinfo=datetime.timezone.utc)}

In [54]:
%%timeit
import random

#fetch a random time from valid_time
options = ds['valid_time'].values

random_time = random.choice(options)
dt = pd.to_datetime(random_time, utc=True)
sun_info = sun.sun(observer, date=dt)
if dt < sun_info['sunrise']:
    print(f"Randomly selected time: {dt} is NIGHTTIME")
elif dt >= sun_info['sunrise'] and dt < sun_info['sunset']:
    print(f"Randomly selected time: {dt} is DAYTIME")
else:
    print(f"Randomly selected time: {dt} is NIGHTTIME")


Randomly selected time: 2024-11-12 07:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-18 04:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-06 02:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-21 08:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-08 05:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-05 01:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-08 09:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-20 23:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-07 21:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-17 12:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-30 00:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-27 03:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-23 01:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-18 07:00:00+00:00 is DAYTIME
Randomly selected time: 2024-11-30 23:00:00+00:00 is NIGHTTIME
Randomly selected time: 2024-11-15 05:00:00+00:00 is DAYTIME
Randomly s

This tells us that we can use the valid time for the specific location of each data point in the query and know based on the sun whether it was daytime or nighttime. Let's put this in a function


In [55]:
#| export
from tqdm import tqdm

def compute_diurnal_class(
        ds: xr.Dataset
    )-> list:
    """
    Compute the diurnal value for each data point in the dataset.
    This function iterates over each data point in the dataset,
    calculates the sunrise and sunset times for the given time, latitude and longitude,
    and returns whether or not that data point is daytime or nighttime.
    """
    diurnal_class = []
    
    for i in tqdm(range(len(ds['valid_time'])), desc="Computing diurnal class"):
        time = ds['valid_time'].values[i]
        dt = pd.to_datetime(time, utc=True)
        observer = Observer(latitude=lat, longitude=long, elevation=0)
        sun_info = sun.sun(observer, date=dt)

        if dt < sun_info['sunrise']:
            diurnal_class.append("night")
        elif dt >= sun_info['sunrise'] and dt < sun_info['sunset']:
            diurnal_class.append("day")
        else:
            diurnal_class.append("night")
    
    return diurnal_class

In [56]:
compute_diurnal_class(ds)

Computing diurnal class:   0%|          | 0/720 [00:00<?, ?it/s]

Computing diurnal class: 100%|██████████| 720/720 [00:00<00:00, 6957.25it/s]


['night',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'night',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'day',
 'night',
 'night',
 'night'

We can then add this value to the dataset as a new variable, which we can use to compute the diurnal averages.

In [57]:
diurnal_vals = compute_diurnal_class(ds)
ds = ds.assign_coords(diurnal_class=("valid_time", diurnal_vals))

Computing diurnal class: 100%|██████████| 720/720 [00:00<00:00, 6920.39it/s]


Now, to see if it will aggregate:

In [58]:
ds.coords["date"] = ds["valid_time"].dt.floor("D")
#ds.groupby(["date", "diurnal_class"]).reduce(np.mean)

In [59]:
print(ds['diurnal_class'].dims)
print(ds['date'].dims)

('valid_time',)
('valid_time',)


In [60]:
ds["d2m"].isel(valid_time=0)
ds["d2m"].values.shape
np.isnan(ds["d2m"].values).sum()

np.int64(0)

In [61]:
aggregated = ds.groupby(["date", "diurnal_class"]).reduce(np.nanmean, dim="valid_time")
#aggregated = ds.groupby(["date", "diurnal_class"]).mean("valid_time")

In [ ]:
aggregated['d2m']

ValueError: Only 1d and 2d plots are supported for facets in xarray. See the package `Seaborn` for more options.